## Create or load the dataset

In [1]:
import pandas as pd

df = pd.DataFrame({
    "Age": [22, 35, 45, 28, 50, 31, 40, 25, 38, 47,
            26, 33, 52, 29, 42, 36, 24, 48, 39, 30],
    "MonthlyCharges": [400, 800, 750, 600, 900, 500, 850, 450, 700, 950,
                       520, 680, 880, 550, 820, 720, 430, 920, 760, 610],
    "Tenure": [3, 24, 36, 8, 48, 12, 30, 5, 20, 52,
               7, 18, 45, 9, 32, 22, 4, 50, 28, 10],
    "Contract": [
        "Monthly", "Annual", "Annual", "Monthly", "Annual",
        "Monthly", "Annual", "Monthly", "Annual", "Annual",
        "Monthly", "Annual", "Annual", "Monthly", "Annual",
        "Annual", "Monthly", "Annual", "Annual", "Monthly"
    ],
    "Churn": [1, 0, 0, 1, 0, 1, 0, 1, 0, 0,
              1, 0, 0, 1, 0, 0, 1, 0, 0, 1]
})

df.head()

,Age,MonthlyCharges,Tenure,Contract,Churn
0,22,400,3,Monthly,1
1,35,800,24,Annual,0
2,45,750,36,Annual,0
3,28,600,8,Monthly,1
4,50,900,48,Annual,0


## Separate features and target

In [3]:
X=df.drop(columns="Churn")
y=df["Churn"]

print("Dataset shape:", df.shape)
print("Features:", X.columns.tolist())
print("Target:", y.name)
print("Class counts:\n", y.value_counts())

Dataset shape: (20, 5)
Features: ['Age', 'MonthlyCharges', 'Tenure', 'Contract']
Target: Churn
Class counts:
 Churn
0    12
1     8
Name: count, dtype: int64


## Split the data

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 15
Testing rows: 5


## Preprocess and train using a Pipeline

A pipeline combines preprocessing and modeling into one reusable workflow.

This pipeline isolates your number columns and performs a two-step sequence:
1. SimpleImputer(strategy="median"): Looks for any missing values (NaN) in these columns and automatically fills them with the median value of that column.
2. StandardScaler(): Standardizes the features by scaling the numbers so they have a mean of 0 and a variance of 1. This prevents columns with massive numbers (like MonthlyCharges) from mathematically dominating columns with smaller numbers (like Age).

• The ColumnTransformer splits your dataset down the middle. It sends your numerical columns to the numeric_pipeline, sends your text columns to the categorical_pipeline simultaneously, processes them in parallel, and then glues the columns back together into one clean, fully processed numeric matrix.
• This is your final Master Pipeline. It takes your combined preprocessor network as step one, and attaches a LogisticRegression classifier as step two.
• When you call model.fit(X_train, y_train), the entire data pipeline executes smoothly from start to finish.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = ["Age", "MonthlyCharges", "Tenure"]
categorical_features = ["Contract"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Train the model

In [8]:
model.fit(X_train, y_train)

print("Training completed.")

Training completed.


## Generate and evaluate predictions

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))

print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(
    y_test, y_pred, zero_division=0
))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Confusion matrix:
 [[3 0]
 [0 2]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



## Make a prediction for a new customer

In [10]:
new_customer = pd.DataFrame({
    "Age": [29],
    "MonthlyCharges": [650],
    "Tenure": [6],
    "Contract": ["Monthly"]
})

prediction = model.predict(new_customer)[0]
probability = model.predict_proba(new_customer)[0, 1]

print("Predicted churn:", "Yes" if prediction == 1 else "No")
print(f"Estimated churn probability: {probability:.2%}")

Predicted churn: Yes
Estimated churn probability: 77.57%


## Once you've selected and evaluated a model, you can save it:

Make sure the models/ directory exists first.

In [18]:
import joblib

joblib.dump(model, "../../models/churn_classifier_v1.joblib")

['../../models/churn_classifier_v1.joblib']

## load it later

In [22]:
loaded_model = joblib.load("../../models/churn_classifier_v1.joblib")

loaded_model.predict(new_customer)

array([1])